## PACOTES ##

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import json

from pathlib import Path
from IPython.display import display

## CÓDIGO ##

In [ ]:
BASE_PATH = Path("thoracic_surgery_base_modelo_statsmodels.csv")
OUTPUT_DIR = Path("outputs")

CAMINHO_MODELO_FINAL = OUTPUT_DIR / "modelo_final.pkl"
CAMINHO_VARIAVEIS_FINAIS = OUTPUT_DIR / "variaveis_finais.json"

CAMINHO_TABELA_OR = OUTPUT_DIR / "tabela_or.csv"

VAR_RESPOSTA = "obito_1_ano"

In [ ]:

dados = pd.read_csv(BASE_PATH)

with open(CAMINHO_VARIAVEIS_FINAIS, "r", encoding="utf-8") as arquivo:
    info_finais = json.load(arquivo)

resultado_final = sm.load(str(CAMINHO_MODELO_FINAL))

print("Modelo final carregado com sucesso.")
print("\nArquivo do modelo:")
print(CAMINHO_MODELO_FINAL)

print("\nArquivo das variáveis finais:")
print(CAMINHO_VARIAVEIS_FINAIS)

print("\nDimensão da base:")
print(dados.shape)

print("\nVariáveis no modelo final:")
display(pd.DataFrame({"variavel": info_finais["variaveis_modelo_final"]}))

Modelo final carregado com sucesso.

Arquivo do modelo:
outputs\modelo_final.pkl

Arquivo das variáveis finais:
outputs\variaveis_finais.json

Dimensão da base:
(470, 26)

Variáveis no modelo final:


,variavel
0,constante
1,"C(diagnostico, Treatment(reference='DGN3'))[T...."
2,"C(diagnostico, Treatment(reference='DGN3'))[T...."
3,"C(diagnostico, Treatment(reference='DGN3'))[T...."
4,"C(diagnostico, Treatment(reference='DGN3'))[T...."
5,"C(diagnostico, Treatment(reference='DGN3'))[T...."
6,"C(diagnostico, Treatment(reference='DGN3'))[T...."
7,"C(tamanho_tumor_tnm, Treatment(reference='OC11..."
8,"C(tamanho_tumor_tnm, Treatment(reference='OC11..."
9,"C(tamanho_tumor_tnm, Treatment(reference='OC11..."


In [ ]:

coeficientes = resultado_final.params

erros_padrao = resultado_final.bse

estatisticas_wald = resultado_final.tvalues

p_valores = resultado_final.pvalues

ic_coef = resultado_final.conf_int(alpha=0.05)
ic_coef.columns = ["ic_95_coef_inferior", "ic_95_coef_superior"]

tabela_or = pd.DataFrame({
    "variavel": coeficientes.index,
    "coeficiente": coeficientes.values,
    "erro_padrao": erros_padrao.values,
    "estatistica_wald_z": estatisticas_wald.values,
    "p_valor": p_valores.values,
    "ic_95_coef_inferior": ic_coef["ic_95_coef_inferior"].values,
    "ic_95_coef_superior": ic_coef["ic_95_coef_superior"].values
})

tabela_or["odds_ratio"] = np.exp(tabela_or["coeficiente"])
tabela_or["ic_95_or_inferior"] = np.exp(tabela_or["ic_95_coef_inferior"])
tabela_or["ic_95_or_superior"] = np.exp(tabela_or["ic_95_coef_superior"])

tabela_or_interpretacao = tabela_or[
    ~tabela_or["variavel"].isin(["const", "constante"])
].copy()

tabela_or_interpretacao = tabela_or_interpretacao.sort_values(
    by="p_valor",
    ascending=True
).reset_index(drop=True)

display(tabela_or_interpretacao)

c:\Users\Vitor Craveiro\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)


,variavel,coeficiente,erro_padrao,estatistica_wald_z,p_valor,ic_95_coef_inferior,ic_95_coef_superior,odds_ratio,ic_95_or_inferior,ic_95_or_superior
0,"C(diagnostico, Treatment(reference='DGN3'))[T....",2.057786,0.572920,3.591748,0.000328,0.934882,3.180689,7.828616e+00,2.546914,24.063329
1,"C(tamanho_tumor_tnm, Treatment(reference='OC11...",1.759053,0.588889,2.987069,0.002817,0.604851,2.913255,5.806937e+00,1.830980,18.416650
2,dispneia_antes_cirurgia,1.142243,0.454220,2.514733,0.011912,0.251987,2.032499,3.133790e+00,1.286580,7.633137
3,tabagismo,1.210157,0.488591,2.476828,0.013256,0.252536,2.167779,3.354012e+00,1.287285,8.738852
4,diabetes_mellitus_tipo_2,1.028661,0.429024,2.397677,0.016499,0.187789,1.869532,2.797316e+00,1.206579,6.485260
5,"C(diagnostico, Treatment(reference='DGN3'))[T....",3.399590,1.507928,2.254478,0.024166,0.444106,6.355074,2.995180e+01,1.559095,575.404681
6,"C(tamanho_tumor_tnm, Treatment(reference='OC11...",1.204909,0.603064,1.997979,0.045719,0.022925,2.386892,3.336454e+00,1.023190,10.879627
7,"C(diagnostico, Treatment(reference='DGN3'))[T....",0.483186,0.402068,1.201752,0.229459,-0.304852,1.271223,1.621231e+00,0.737232,3.565212
8,"C(tamanho_tumor_tnm, Treatment(reference='OC11...",0.381487,0.317777,1.200487,0.229950,-0.241344,1.004317,1.464460e+00,0.785571,2.730043
9,"C(diagnostico, Treatment(reference='DGN3'))[T....",0.323904,0.456098,0.710162,0.477603,-0.570032,1.217840,1.382514e+00,0.565507,3.379878


In [5]:

def interpretar_or(or_valor):
    if or_valor > 1:
        return "Aumento da chance de óbito"
    elif or_valor < 1:
        return "Redução da chance de óbito"
    else:
        return "Sem alteração na chance de óbito"


def interpretar_significancia(p_valor, alpha=0.05):
    if p_valor < alpha:
        return "Estatisticamente significativa ao nível de 5%"
    else:
        return "Não significativa ao nível de 5%"


tabela_or_interpretacao["interpretacao_or"] = tabela_or_interpretacao["odds_ratio"].apply(
    interpretar_or
)

tabela_or_interpretacao["significancia"] = tabela_or_interpretacao["p_valor"].apply(
    interpretar_significancia
)

display(tabela_or_interpretacao)

,variavel,coeficiente,erro_padrao,estatistica_wald_z,p_valor,ic_95_coef_inferior,ic_95_coef_superior,odds_ratio,ic_95_or_inferior,ic_95_or_superior,interpretacao_or,significancia
0,"C(diagnostico, Treatment(reference='DGN3'))[T....",2.057786,0.572920,3.591748,0.000328,0.934882,3.180689,7.828616e+00,2.546914,24.063329,Aumento da chance de óbito,Estatisticamente significativa ao nível de 5%
1,"C(tamanho_tumor_tnm, Treatment(reference='OC11...",1.759053,0.588889,2.987069,0.002817,0.604851,2.913255,5.806937e+00,1.830980,18.416650,Aumento da chance de óbito,Estatisticamente significativa ao nível de 5%
2,dispneia_antes_cirurgia,1.142243,0.454220,2.514733,0.011912,0.251987,2.032499,3.133790e+00,1.286580,7.633137,Aumento da chance de óbito,Estatisticamente significativa ao nível de 5%
3,tabagismo,1.210157,0.488591,2.476828,0.013256,0.252536,2.167779,3.354012e+00,1.287285,8.738852,Aumento da chance de óbito,Estatisticamente significativa ao nível de 5%
4,diabetes_mellitus_tipo_2,1.028661,0.429024,2.397677,0.016499,0.187789,1.869532,2.797316e+00,1.206579,6.485260,Aumento da chance de óbito,Estatisticamente significativa ao nível de 5%
5,"C(diagnostico, Treatment(reference='DGN3'))[T....",3.399590,1.507928,2.254478,0.024166,0.444106,6.355074,2.995180e+01,1.559095,575.404681,Aumento da chance de óbito,Estatisticamente significativa ao nível de 5%
6,"C(tamanho_tumor_tnm, Treatment(reference='OC11...",1.204909,0.603064,1.997979,0.045719,0.022925,2.386892,3.336454e+00,1.023190,10.879627,Aumento da chance de óbito,Estatisticamente significativa ao nível de 5%
7,"C(diagnostico, Treatment(reference='DGN3'))[T....",0.483186,0.402068,1.201752,0.229459,-0.304852,1.271223,1.621231e+00,0.737232,3.565212,Aumento da chance de óbito,Não significativa ao nível de 5%
8,"C(tamanho_tumor_tnm, Treatment(reference='OC11...",0.381487,0.317777,1.200487,0.229950,-0.241344,1.004317,1.464460e+00,0.785571,2.730043,Aumento da chance de óbito,Não significativa ao nível de 5%
9,"C(diagnostico, Treatment(reference='DGN3'))[T....",0.323904,0.456098,0.710162,0.477603,-0.570032,1.217840,1.382514e+00,0.565507,3.379878,Aumento da chance de óbito,Não significativa ao nível de 5%


In [6]:
tabela_or_interpretacao.to_csv(
    CAMINHO_TABELA_OR,
    index=False,
    encoding="utf-8-sig"
)

print("Tabela de odds ratios salva em:")
print(CAMINHO_TABELA_OR)

Tabela de odds ratios salva em:
outputs\tabela_or.csv
